# RAG From Scratch: Evaluation

Modern RAG work should start with evals. This notebook separates retrieval quality from answer quality so changes to chunking, embeddings, reranking, prompts, or models can be compared instead of eyeballed.

References: RAGAS, ARES, DeepEval, and recent RAG evaluation surveys.

## Metrics map

- Retrieval: recall@k, precision@k, MRR, nDCG, context precision, context recall.
- Generation: answer relevance, faithfulness/groundedness, citation correctness, refusal correctness.
- System: latency, token cost, index freshness, failure rate.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
from collections import defaultdict
import math

gold_relevant = {
    "q1": {"doc:chunking", "doc:evals"},
    "q2": {"doc:hybrid"},
    "q3": {"doc:versions", "doc:acl"},
}

ranked_results = {
    "q1": ["doc:evals", "doc:chunking", "doc:hybrid"],
    "q2": ["doc:graph", "doc:hybrid", "doc:rerank"],
    "q3": ["doc:versions", "doc:hybrid", "doc:acl"],
}

def recall_at_k(results, relevant, k):
    hits = set(results[:k]) & relevant
    return len(hits) / max(len(relevant), 1)

def precision_at_k(results, relevant, k):
    hits = set(results[:k]) & relevant
    return len(hits) / max(k, 1)

def reciprocal_rank(results, relevant):
    for rank, doc_id in enumerate(results, start=1):
        if doc_id in relevant:
            return 1 / rank
    return 0

def dcg(results, relevant, k):
    return sum((1 if doc_id in relevant else 0) / math.log2(i + 2) for i, doc_id in enumerate(results[:k]))

def ndcg_at_k(results, relevant, k):
    ideal = dcg(list(relevant), relevant, min(k, len(relevant)))
    return dcg(results, relevant, k) / ideal if ideal else 0

def evaluate_retrieval(ranked_results, gold_relevant, k=3):
    rows = []
    for qid, relevant in gold_relevant.items():
        results = ranked_results.get(qid, [])
        rows.append({
            "qid": qid,
            f"recall@{k}": recall_at_k(results, relevant, k),
            f"precision@{k}": precision_at_k(results, relevant, k),
            "mrr": reciprocal_rank(results, relevant),
            f"ndcg@{k}": ndcg_at_k(results, relevant, k),
        })
    return rows

rows = evaluate_retrieval(ranked_results, gold_relevant, k=3)
rows

In [ ]:
def summarize(rows):
    metrics = [key for key in rows[0] if key != "qid"]
    return {metric: sum(row[metric] for row in rows) / len(rows) for metric in metrics}

summarize(rows)

## Optional Ragas evaluation

Use this after you have a real RAG chain and API keys configured. Ragas can evaluate context precision/recall, faithfulness, and answer relevancy over a small benchmark dataset.

In [ ]:
# Optional example; uncomment after configuring your model provider.
# from ragas import EvaluationDataset, evaluate
# from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextRecall
#
# dataset = EvaluationDataset.from_list([
#     {
#         "user_input": "When should I prefer hybrid retrieval?",
#         "retrieved_contexts": ["Hybrid retrieval combines sparse keyword search and dense semantic search."],
#         "response": "Prefer hybrid retrieval when exact terms, numbers, entities, or rare keywords matter.",
#         "reference": "Hybrid retrieval is useful when lexical matches and semantic matches are both needed.",
#     }
# ])
# result = evaluate(dataset, metrics=[Faithfulness(), ResponseRelevancy(), LLMContextRecall()])
# result